# Differentiable-FEA — Google Colab (GPU) edition

Runs the project on Colab's free GPU: **Runtime → Change runtime type → T4 GPU**, then *Run all*.

### Security — read once
**What this notebook does and does not access.** It is written so that nothing in it can reach your Google account or files:

* it **never** calls `drive.mount`, `auth.authenticate_user`, or reads Colab *Secrets* — do not add such cells;
* the only network access is `git fetch` of the **public** project repository at a **pinned commit** (a fixed hash, so the code cannot change under you) and `pip install` of two version-pinned packages;
* the optional GUI is shown **inside the notebook** through Colab's own authenticated proxy — **no ngrok/localtunnel/public URL** is created;
* everything lives in the VM's temporary `/content` disk, which is deleted when the runtime ends.

Rules to keep it that way: run only notebooks you wrote or fully read; never paste passwords/tokens into cells; don't mount Drive "for convenience" (a notebook that can read Drive can read your Drive). For extra safety use a dedicated Google account for experiments.

## 1  Check the GPU

In [ ]:
import platform, torch
print("PyTorch", torch.__version__, "| Python", platform.python_version())
HAS_GPU = torch.cuda.is_available()
print("CUDA available:", HAS_GPU)
if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -> Runtime > Change runtime type > T4 GPU. (The notebook still runs on the CPU.)")

## 2  Fetch the code (public repo, pinned commit) and install two pinned packages


In [ ]:
import os, re, shutil, subprocess

REPO = "https://github.com/miladshirani/Differentiable-FEA.git"   # <-- the PUBLIC repository
REF  = "ddbec4ef03766e95a6860f92166f28d6374386b2"   # <-- pin: the full commit hash you want to run (git rev-parse HEAD)
ROOT = "/content/difffea"                 # temporary VM disk, deleted when the runtime ends
CODE = ROOT

assert "<" not in REPO + REF, "Edit REPO and REF at the top of this cell first."
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit hash (never a branch name)."

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)     # check=True: a failing step stops here with a clear message

shutil.rmtree(ROOT, ignore_errors=True)
sh(f"git init -q {ROOT}")
sh(f"git -C {ROOT} remote add origin {REPO}")
sh(f"git -C {ROOT} fetch -q --depth 1 origin {REF}")      # fetch exactly the pinned commit
sh(f"git -C {ROOT} checkout -q FETCH_HEAD")
sh("pip install -q gmsh==4.15.2 streamlit==1.65.0")            # the only two third-party packages installed (pinned)

assert os.path.isdir(CODE), f"code folder not found: {CODE}"
print("code downloaded to", CODE)

In [ ]:
import sys, os
os.chdir(CODE)
sys.path.insert(0, CODE + "/src")                  # the library comes straight from the pinned checkout (no build step)
os.environ["PYTHONPATH"] = CODE + "/src"           # child processes (benchmarks, GUI) find it too
import difffea.problem
print("code loaded from", os.path.dirname(difffea.problem.__file__))

## 3  Benchmark: CPU vs GPU (the measurements of roadmap step 2)
`benchmarks/bench_device.py` times the residual and the matrix-free tangent product (three evaluations: `jvp`, `linearize`, `qp`) on a growing mesh and runs one complete nonlinear solve, on the CPU and on the GPU. On the GPU it also compares the solution with the CPU solution (correctness). All arithmetic is float64: the free T4 has weak float64 units, but only the measurement tells how much that matters. The last line of each run is a JSON block with every number; copy it when you report results.


In [ ]:
sh(f"python {CODE}/benchmarks/bench_device.py --device cpu")
if HAS_GPU:
    sh(f"python {CODE}/benchmarks/bench_device.py --device cuda")
else:
    print("No GPU: switch the runtime to a T4 GPU to measure the GPU.")

### Where does the time go? (profiler)
`benchmarks/profile_matvec.py` breaks one tangent product down by operator and compares it with the memory-bandwidth limit of the device.


In [ ]:
sh(f"python {CODE}/benchmarks/profile_matvec.py --device {'cuda' if HAS_GPU else 'cpu'} --elements 40000")

## 4  Solve a nonlinear problem on the GPU
Plate with a circular hole under tension, neo-Hookean, Q4 elements. We compare preconditioners and devices.
`ilu` assembles a sparse matrix and runs on the CPU (best on CPU); `jacobi` and `block_jacobi` are matrix-free and run entirely on the GPU.

In [ ]:
import time, torch
from difffea.problem import default_spec, build_problem
from difffea.newton_krylov import newton_krylov_solve
from difffea.postprocess import compute_fields

H = 0.06   # element size: smaller = bigger problem

def sync(device):
    if device == "cuda":
        torch.cuda.synchronize()

def run(device, preconditioner):
    spec = default_spec("plate_hole", "quad4", H)
    spec["device"] = device
    p = build_problem(spec)
    theta = torch.ones(p["conn"].shape[0], dtype=torch.float64, device=device)
    tol = 1e-8 * float(torch.norm(p["f"]))
    t = time.time()
    u, info = newton_krylov_solve(p, theta, 3, tol, verbose=False, preconditioner=preconditioner, tangent="qp")
    sync(device)
    return p, theta, u, info, time.time() - t

results = {}
for device, pc in [("cpu", "jacobi"), ("cpu", "ilu")] + ([("cuda", "jacobi"), ("cuda", "block_jacobi")] if HAS_GPU else []):
    p, theta, u, info, secs = run(device, pc)
    results[(device, pc)] = (p, theta, u)
    print(f"{device:5s} {pc:13s} elements={p['conn'].shape[0]:6d} converged={info['converged']} "
          f"Newton={info['total_newton']:3d} CG={info['total_cg']:6d}  {secs:7.1f} s")

ref = results[("cpu", "ilu")][2].cpu()
for key, (_, _, u) in results.items():
    print(key, "max |u - u_ref| =", float((u.cpu() - ref).abs().max()))

In [ ]:
import numpy as np
from matplotlib.figure import Figure
from difffea.plotting import plot_field
p, theta, u = results[("cpu", "ilu")]
f = compute_fields(u, theta, p)
fig = Figure(figsize=(9, 4.5)); ax = fig.subplots()
pc = plot_field(ax, p["nodes"].numpy(), p["conn"].numpy(), "quad4", f["mises_n"].numpy(), u.numpy(), 1.0, cmap="inferno", show_edges=False)
fig.colorbar(pc, ax=ax, label="von Mises"); ax.set_title("von Mises stress (deformed shape)")
fig

## 5  (Optional) the interactive GUI inside the notebook
Starts the Streamlit app bound to **localhost only** and displays it through Colab's authenticated proxy (visible only in your own session). No public tunnel is created — do **not** replace this with ngrok or localtunnel.
When a GPU is available the app shows a *Compute device* selector.

In [ ]:
import subprocess, time, sys
from google.colab import output

server = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py",
     "--server.address", "127.0.0.1", "--server.port", "8501", "--server.headless", "true",
     "--browser.gatherUsageStats", "false",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],   # needed behind Colab's proxy; the port is not exposed publicly
    cwd=CODE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
output.serve_kernel_port_as_iframe(8501, height=900)

In [ ]:
# Stop the GUI server when you are done
server.terminate()